# Demo page export: the recorded test trajectories, for GitHub Pages

**Runtime:** CPU (Runtime, Change runtime type, CPU). No GPU, no model and no search server: this notebook only reads the saved evaluation outputs in Drive. It uses almost no compute units.

**What it does.** Reads `data/<set>_test.jsonl` and the saved test trajectories in `outputs/`, the same files the evaluation notebook read, and writes the data behind the static demo page: `meta.json` and one `q_<set>.json` per test set. Nothing is rerun, and nothing in `data/`, `outputs/` or `checkpoints/` changes. The files go to `site_data/` in the project folder, zipped as `site_data.zip`, and the zip downloads to your computer at the end.

**Policies on the page:** the untrained agent, the start policy, the start policy with one search (RAG), and the four rewards trained from it: plain GRPO, fixed penalty (lambda 0.05), SAAS and Soft-SAAS, seed 0 each. Every question also shows the start policy's closed book answer and its p(q).

**Gate.** Every recomputed EM, F1 and search count for those policies, and for the start policy's closed book answers, matches the report's appendix table on all five test sets; the known question counts match the report's adaptivity table; no question is missing a policy.

## Storage

In [ ]:
PROJECT_NAME = "search_or_think"

import json
import os
import shutil
import sys

CODE = "/content/code"
os.makedirs(f"{CODE}/sot", exist_ok=True)
sys.path.insert(0, CODE)

from google.colab import drive
drive.mount("/content/drive", timeout_ms=90_000)
PROJECT = f"/content/drive/MyDrive/{PROJECT_NAME}"
DATA, OUT = f"{PROJECT}/data", f"{PROJECT}/outputs"
assert os.path.isdir(OUT), f"no outputs/ under {PROJECT}: is this the Google account that has the project?"
SITE = "/content/site_data"
print("project:", PROJECT, "| test sets:", sorted(f for f in os.listdir(DATA) if f.endswith("_test.jsonl")))

## Code

`sot/scoring.py`, `sot/rewards.py` and `sot/analysis.py` are the evaluation notebook's files, unchanged: the page's numbers come from the same functions as the report's.

In [ ]:
%%writefile /content/code/sot/__init__.py
"""Search or Think? project package."""

In [ ]:
%%writefile /content/code/sot/scoring.py
"""Answer scoring (SQuAD / HotpotQA style EM and F1) and evidence matching."""
import collections
import re
import string
import unicodedata

_PUNCT = set(string.punctuation)


def normalize_answer(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def em(pred, golds):
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in golds))


def f1(pred, golds):
    p = normalize_answer(pred)
    best = 0.0
    for g in golds:
        g = normalize_answer(g)
        # HotpotQA official eval: yes/no answers only score on an exact match
        if p != g and ({p, g} & {"yes", "no", "noanswer"}):
            continue
        common = collections.Counter(p.split()) & collections.Counter(g.split())
        ns = sum(common.values())
        if ns:
            prec, rec = ns / len(p.split()), ns / len(g.split())
            best = max(best, 2 * prec * rec / (prec + rec))
    return best


def has_answer(text, golds):
    t = f" {normalize_answer(text)} "
    return any(f" {normalize_answer(g)} " in t for g in golds if normalize_answer(g))


def norm_title(t):
    return unicodedata.normalize("NFKC", t or "").strip().strip('"').lower()


def norm_query(q):
    """Two queries that differ only in case or spacing are the same query: BM25 returns the same passages."""
    return " ".join((q or "").lower().split())


def n_distinct(queries):
    """Distinct queries in one episode. A repeated query retrieves nothing new, so this is how many retrievals
    the episode really used; N_min is estimated from it (see analysis.boundary_table)."""
    return len({norm_query(q) for q in queries})


def knows(answer, golds):
    """The correctness test behind p(q): cover match, i.e. a gold answer is contained in the closed book answer.
    Lenient on form because p(q) measures knowledge, and strict EM misses verbose correct answers (interim report)."""
    return has_answer(answer or "", golds)

In [ ]:
%%writefile /content/code/sot/rewards.py
"""Rewards: plain GRPO, fixed per search penalty, SAAS, and Soft-SAAS (Eq. 1 of the proposal).

One definition per quantity, used in training, relabelling and analysis alike:
  p(q)   share of the K search disabled rollouts whose answer covers a gold answer (scoring.knows, cover match)
  N_min  fewest searches among search enabled rollouts with F1 = 1 on the answer inside <answer> tags (is_correct)
  gate   a search penalty applies only to rollouts with F1 = 1 on the answer inside <answer> tags (is_correct)
SAAS and Soft-SAAS read the same p(q) and N_min, so the hard vs soft boundary is the only difference between them.
The released SAAS code uses EM for its labels; we keep cover match for p(q) and store an EM based p(q) next to it.
"""
from dataclasses import dataclass
from typing import Optional

MAX_SEARCHES = 4
ALPHA = 0.05  # SAAS: alpha in the paper (App. B.1), beta = 0.05 in search/utils.py of the released code

# NoSearch needs delta of K search disabled rollouts correct. SAAS uses delta = 2 of 4 in the paper (Sec. 3.1) and in
# the released code (head_correct_count >= 2 with disable_search_first_n = 4); as a ratio it carries over to K = 8.
NOSEARCH_MIN_P = 0.5
# The paper and the released code disagree on NeedSearch. Paper: no search disabled rollout is correct and some
# search enabled one is. Code (generate_with_search.py, mq_group_reward_func): fewer than delta are correct and some
# search enabled one is. We follow the code by default; "paper" stays available for the comparison.
SAAS_RULE = "code"


def is_correct(f1):
    return f1 >= 1.0 - 1e-9


@dataclass
class Boundary:
    p: float              # p(q)
    n_min: Optional[int]  # N_min (None if no search enabled rollout was correct)
    label: str            # SAAS label: NoSearch / NeedSearch / Undetermined


def saas_label(p, n_min, rule=SAAS_RULE, nosearch_min_p=NOSEARCH_MIN_P):
    if rule not in ("code", "paper"):
        raise ValueError(f"unknown SAAS rule {rule!r}")
    if p >= nosearch_min_p:
        return "NoSearch"
    if n_min is not None and (rule == "code" or p == 0):
        return "NeedSearch"
    return "Undetermined"


def estimate_boundary(disabled_correct, enabled_rollouts, rule=SAAS_RULE, nosearch_min_p=NOSEARCH_MIN_P):
    """disabled_correct: list[bool] (cover match); enabled_rollouts: list of (correct: bool, n_search: int) (F1 = 1)."""
    p = sum(disabled_correct) / len(disabled_correct) if disabled_correct else 0.0
    ns = [n for c, n in enabled_rollouts if c]
    n_min = min(ns) if ns else None
    return Boundary(p, n_min, saas_label(p, n_min, rule, nosearch_min_p))


def penalty_saas(n_search, b):
    if b.label == "NoSearch":
        return float(n_search)
    if b.label == "NeedSearch":
        return float(max(0, n_search - b.n_min))
    return 0.0  # Undetermined questions get no search signal


def penalty_soft(n_search, b, max_searches=MAX_SEARCHES):
    n_min = max_searches if b.n_min is None else b.n_min
    return b.p * n_search + (1 - b.p) * max(0, n_search - n_min)


def reward_plain(f1, n_search):
    return f1


def reward_fixed(f1, n_search, lam):
    return f1 - lam * n_search


def reward_saas(f1, n_search, b, alpha, w=1.0):
    return f1 - (alpha * w * penalty_saas(n_search, b) if is_correct(f1) else 0.0)


def reward_soft_saas(f1, n_search, b, alpha, w=1.0, max_searches=MAX_SEARCHES):
    return f1 - (alpha * w * penalty_soft(n_search, b, max_searches) if is_correct(f1) else 0.0)


def compute_rewards(kind, f1s, n_searches, boundaries, *, lam=0.1, alpha=ALPHA, w=1.0, format_bonus=0.0,
                    format_ok=None):
    """Rewards for a batch of rollouts. Returns (rewards, penalised), penalised[i] = rollout i lost reward for searching.
    format_bonus: reward for a well formed but wrong answer (F1 = 0), as in the released SAAS code (0.1 there)."""
    rewards, penalised = [], []
    for i, (f, n) in enumerate(zip(f1s, n_searches)):
        if kind == "plain":
            pen = 0.0
        elif kind == "fixed":
            pen = lam * w * n
        elif kind == "saas":
            pen = alpha * w * penalty_saas(n, boundaries[i]) if is_correct(f) else 0.0
        elif kind == "soft":
            pen = alpha * w * penalty_soft(n, boundaries[i]) if is_correct(f) else 0.0
        else:
            raise ValueError(f"unknown reward {kind!r}")
        bonus = format_bonus if format_bonus and f == 0 and format_ok is not None and format_ok[i] else 0.0
        rewards.append(f + bonus - pen)
        penalised.append(pen > 0)
    return rewards, penalised


def tighten_n_min(boundaries, f1s, n_searches, G, rule=SAAS_RULE):
    """On-policy N_min, as in the released SAAS code: within each group of G rollouts of one question, N_min is at
    most the fewest searches of the group's own correct rollouts. p(q) is left as it was measured."""
    out = list(boundaries)
    for g in range(0, len(out), G):
        ns = [n for f, n in zip(f1s[g:g + G], n_searches[g:g + G]) if is_correct(f)]
        if not ns:
            continue
        for i in range(g, min(g + G, len(out))):
            b = out[i]
            n_min = min(ns) if b.n_min is None else min(b.n_min, min(ns))
            out[i] = Boundary(b.p, n_min, saas_label(b.p, n_min, rule))
    return out


def group_stats(rewards, G):
    """Share of question groups (G consecutive rollouts) with no reward variance, which give GRPO no gradient."""
    groups = [rewards[g:g + G] for g in range(0, len(rewards), G)]
    flat = [max(x) - min(x) < 1e-8 for x in groups]
    n = max(len(groups), 1)
    return {"zero_var_share": sum(flat) / n,
            "zero_var_all_zero": sum(f and abs(x[0]) < 1e-8 for f, x in zip(flat, groups)) / n,
            "flat_groups": [i for i, f in enumerate(flat) if f]}


class WarmupTrigger:
    """Self triggered warm up: w(t) = 0 until the moving average of training reward improves by
    less than eps over `window` steps, then w rises linearly to 1 over `ramp` steps.

    Defaults: a 20 step mean that gains less than 0.01 over 40 steps. The interim report's rule (10 step mean,
    under 0.005 over 20 steps) is below the noise of 64 rollouts per step and fires while reward still rises: it
    did so at step 36 of a stage 0 sweep run. OLD_TRIGGER keeps it for the comparison in the report."""

    def __init__(self, window=40, eps=0.01, ramp=50, ma=20):
        self.window, self.eps, self.ramp, self.ma = window, eps, ramp, ma
        self.rewards, self.mas, self.trigger_step = [], [], None

    def update(self, reward_mean):
        self.rewards.append(reward_mean)
        recent = self.rewards[-self.ma:]
        self.mas.append(sum(recent) / len(recent))
        t = len(self.rewards) - 1
        if self.trigger_step is None and len(self.mas) > self.window + self.ma:
            if self.mas[-1] - self.mas[-1 - self.window] < self.eps:
                self.trigger_step = t
        return self.weight(t)

    def weight(self, t):
        if self.trigger_step is None:
            return 0.0
        return max(0.0, min(1.0, (t - self.trigger_step) / self.ramp))


OLD_TRIGGER = dict(window=20, eps=0.005, ma=10)


def replay_trigger(rewards, **kw):
    """Step (1-based, as in log.jsonl) after which the trigger fires on a logged reward curve, or None."""
    t = WarmupTrigger(**kw)
    for r in rewards:
        t.update(r)
    return None if t.trigger_step is None else t.trigger_step + 1


class Warmup:
    """w(t) per optimizer step, keyed by the trainer's global_step.

    spec  "off"            w = 1 from the first step (the penalty is simply on, e.g. a run branched at the switch)
          "manual_step:N"  w = 0 before step N, then a linear ramp to 1 over `ramp` steps (ramp 0 = SAAS's hard switch)
          "self"           WarmupTrigger on the training reward
    observe() may be called many times within one optimizer step (reward calls, accumulation micro-batches,
    resampling); a step's mean reward is committed to the trigger exactly once, when a later step asks for its weight.
    The trigger is tracked for every spec, so any run's log shows when the self trigger would have fired."""

    def __init__(self, spec="off", ramp=None, **trigger_kw):
        self.spec = spec
        self.manual = int(spec.split(":", 1)[1]) if spec.startswith("manual_step:") else None
        if spec not in ("off", "self") and self.manual is None:
            raise ValueError(f"unknown warm up spec {spec!r}")
        self.ramp = (50 if spec == "self" else 0) if ramp is None else ramp
        self.trigger = WarmupTrigger(ramp=max(self.ramp, 1), **trigger_kw)
        self.pending = {}  # step -> [reward sum, count], not yet committed

    @property
    def trigger_step(self):
        return self.trigger.trigger_step

    def observe(self, step, reward_mean):
        s = self.pending.setdefault(step, [0.0, 0])
        s[0] += reward_mean
        s[1] += 1

    def _commit(self, upto):
        while len(self.trigger.rewards) < upto:
            t = len(self.trigger.rewards)
            if t in self.pending:
                total, count = self.pending.pop(t)
                self.trigger.update(total / count)
            else:  # a step without its own rollouts reuses the last reward
                self.trigger.update(self.trigger.rewards[-1] if self.trigger.rewards else 0.0)

    def weight(self, step):
        self._commit(step)  # w(step) only depends on the rewards of earlier steps
        if self.spec == "off":
            return 1.0
        if self.manual is not None:
            if step < self.manual:
                return 0.0
            return 1.0 if self.ramp == 0 else min(1.0, (step - self.manual) / self.ramp)
        return self.trigger.weight(step)

    def state(self):
        rewards = list(self.trigger.rewards)
        while len(rewards) in self.pending:  # steps observed but not yet committed
            total, count = self.pending[len(rewards)]
            rewards.append(total / count)
        return {"spec": self.spec, "ramp": self.ramp, "rewards": rewards, "trigger_step": self.trigger_step}

    def restore(self, state, upto):
        """Replay the saved reward history of steps < upto (the step training resumes from)."""
        self.trigger.rewards, self.trigger.mas, self.trigger.trigger_step = [], [], None
        self.pending = {}
        for r in state["rewards"][:upto]:
            self.trigger.update(r)

In [ ]:
%%writefile /content/code/sot/analysis.py
"""Metrics over saved runs: accuracy tables, p(q) distribution, QOR, SOR, and the SAAS vs Soft-SAAS signal check."""
import collections
import json
import math
import random

from .rewards import SAAS_RULE, Boundary, estimate_boundary, is_correct, penalty_saas, penalty_soft
from .scoring import has_answer, knows, norm_query, norm_title


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else math.nan


def by_dataset(rows):
    g = collections.defaultdict(list)
    for r in rows:
        g[r["dataset"]].append(r)
    return dict(sorted(g.items()))


def summarize(rows):
    return {ds: {"n": len(v), "em": mean(r["em"] for r in v), "f1": mean(r["f1"] for r in v),
                 "searches": mean(r["n_search"] for r in v), "format_ok": mean(r["format_ok"] for r in v)}
            for ds, v in by_dataset(rows).items()}


def p_by_id(closed_rows, col="em"):
    """col: "em" (strict) or "cover" (gold answer contained in the prediction)."""
    d = collections.defaultdict(list)
    for r in closed_rows:
        d[r["id"]].append(r[col] >= 1.0)
    return {k: sum(v) / len(v) for k, v in d.items()}


def enriched_sample(recs, pmap, n, mid_frac=1 / 3, seed=0):
    """n questions with about mid_frac at 0 < p(q) < 1 (fewer if the pool has fewer); the rest keep the
    pool's own mix of p(q) = 0 and p(q) = 1."""
    rng = random.Random(seed)
    mid = [r for r in recs if 0 < pmap[r["id"]] < 1]
    rest = [r for r in recs if not 0 < pmap[r["id"]] < 1]
    rng.shuffle(mid)
    rng.shuffle(rest)
    k = min(len(mid), round(n * mid_frac))
    out = mid[:k] + rest[:n - k]
    out += mid[k:k + n - len(out)]  # too few p(q) = 0 or 1 questions: top up with in-between ones
    rng.shuffle(out)
    return out


def p_distribution(pmap, recs):
    out = {}
    for ds, v in by_dataset(recs).items():
        ps = [pmap[r["id"]] for r in v if r["id"] in pmap]
        out[ds] = {"n": len(ps), "hist": dict(sorted(collections.Counter(round(p, 2) for p in ps).items())),
                   "p0": mean(p == 0 for p in ps), "mid": mean(0 < p < 1 for p in ps), "p1": mean(p == 1 for p in ps)}
    return out


def qor(agent_rows, pmap, thr=0.5):
    """Question level over search: share of questions the base model can answer without search
    (p(q) >= thr) on which the policy still searches. Next to it, the search rate on the other questions: a policy
    that knows when to search has qor well below search_rate_rest; one with a fixed habit has the two equal."""
    out = {}
    for ds, v in by_dataset(agent_rows).items():
        elig = [r for r in v if pmap.get(r["id"], 0.0) >= thr]
        rest = [r for r in v if r["id"] in pmap and pmap[r["id"]] < thr]
        out[ds] = {"eligible": len(elig), "qor": mean(r["n_search"] > 0 for r in elig),
                   "search_rate_rest": mean(r["n_search"] > 0 for r in rest),
                   "f1_eligible": mean(r["f1"] for r in elig)}
    return out


def _evidence_complete(rec, titles, text):
    """All gold evidence retrieved: every supporting page title when the set has them (HotpotQA, and 2Wiki and
    MuSiQue when their metadata carries them), otherwise the answer string (NQ, PopQA)."""
    if rec.get("sup_titles"):
        return {norm_title(t) for t in rec["sup_titles"]} <= titles
    if rec["dataset"] == "hotpotqa":
        return False
    return has_answer(text, rec["golds"])


def sor(agent_rows, recs_by_id):
    """Step level over search: share of searches issued after all gold evidence was already retrieved.
    complete: share of episodes whose searches retrieved all gold evidence at some point (a sanity check: near 0 on
    a set with supporting titles means those titles do not match the retrieval corpus)."""
    out = {}
    for ds, v in by_dataset(agent_rows).items():
        over = total = complete = 0
        for r in v:
            rec, titles, text = recs_by_id[r["id"]], set(), ""
            for s in r["searches"]:
                over += _evidence_complete(rec, titles, text)
                total += 1
                titles |= {norm_title(t) for t in s["titles"]}
                text += " " + " ".join(s["texts"])
            complete += bool(r["searches"]) and _evidence_complete(rec, titles, text)
        out[ds] = {"searches": total, "sor": over / total if total else math.nan,
                   "complete": complete / len(v) if v else math.nan}
    return out


def boundary_table(closed_rows, agent_rows, golds_by_id, rule=SAAS_RULE, count="n_search"):
    """Per question knowledge boundary from saved rollouts, with the definitions fixed in rewards.py:
    p(q) = share of search disabled rows whose answer covers a gold answer (p_em is the strict EM version, kept
    for comparison); N_min = fewest searches among search enabled rows that are well formed and have F1 = 1.
    count: the row field that counts an episode's searches. "n_search" counts every call; "n_distinct" counts
    distinct queries, so an episode that repeats one query three times shows the question needs one retrieval.
    Rows come from run_inference.py. Returns {id: {p, p_em, n_min, label, k_closed, k_agent, dataset}}."""
    disabled, disabled_em, enabled, ds_of = (collections.defaultdict(list) for _ in range(4))
    for r in closed_rows:
        disabled[r["id"]].append(knows(r["answer"], golds_by_id[r["id"]]))
        disabled_em[r["id"]].append(r["em"] >= 1.0)
    for r in agent_rows:
        enabled[r["id"]].append((bool(r["format_ok"]) and is_correct(r["f1"]), r[count]))
        ds_of[r["id"]] = r["dataset"]
    out = {}
    for qid in disabled:
        b = estimate_boundary(disabled[qid], enabled.get(qid, []), rule)
        out[qid] = {"p": b.p, "p_em": mean(disabled_em[qid]), "n_min": b.n_min, "label": b.label,
                    "k_closed": len(disabled[qid]), "k_agent": len(enabled.get(qid, [])), "dataset": ds_of.get(qid)}
    return out


def boundary_signal(closed_rows, sampled_agent_rows, golds_by_id, rule=SAAS_RULE, count="n_search", table=None):
    """How many questions can each reward penalise at all, and how often correct searching rollouts get penalised.
    count picks the N_min estimate (see boundary_table); the penalty itself always counts every search call.
    table: a boundary table built elsewhere (e.g. with budget probe rows added), applied to these rollouts."""
    table = table or boundary_table(closed_rows, sampled_agent_rows, golds_by_id, rule, count)
    out = collections.defaultdict(lambda: {"questions": 0, "labels": collections.Counter(), "signal_saas": 0,
                                           "signal_soft": 0, "correct_searching": 0, "pen_saas": [], "pen_soft": []})
    seen = set()
    for r in sampled_agent_rows:
        if r["id"] not in table:
            continue
        t = table[r["id"]]
        b = Boundary(t["p"], t["n_min"], t["label"])
        o = out[r["dataset"]]
        if r["id"] not in seen:
            seen.add(r["id"])
            o["questions"] += 1
            o["labels"][b.label] += 1
            o["signal_saas"] += b.label != "Undetermined"
            o["signal_soft"] += b.p > 0 or b.n_min is not None
        if r["format_ok"] and is_correct(r["f1"]) and r["n_search"] > 0:
            o["correct_searching"] += 1
            o["pen_saas"].append(penalty_saas(r["n_search"], b))
            o["pen_soft"].append(penalty_soft(r["n_search"], b))
    return {ds: {"questions": o["questions"], "labels": dict(o["labels"]),
                 "signal_saas": o["signal_saas"] / max(o["questions"], 1),
                 "signal_soft": o["signal_soft"] / max(o["questions"], 1),
                 "correct_searching_rollouts": o["correct_searching"],
                 "penalised_saas": mean(p > 0 for p in o["pen_saas"]),
                 "penalised_soft": mean(p > 0 for p in o["pen_soft"]),
                 "mean_pen_saas": mean(o["pen_saas"]), "mean_pen_soft": mean(o["pen_soft"])}
            for ds, o in sorted(out.items())}


def seed_averaged_paired(a_runs, b_runs, metric, n_boot=2000):
    """b minus a over several seeds: each question's metric is first averaged over the runs of each side (only
    questions present in every run), then a paired bootstrap over questions. a_runs, b_runs: lists of row lists."""
    def avg(runs):
        d = collections.defaultdict(list)
        for rows in runs:
            for r in rows:
                d[r["id"]].append(r[metric])
        return {k: mean(v) for k, v in d.items() if len(v) == len(runs)}
    a, b = avg(a_runs), avg(b_runs)
    keys = sorted(set(a) & set(b))
    return {**paired_bootstrap([a[k] for k in keys], [b[k] for k in keys], n_boot), "n": len(keys)}


def paired_bootstrap(a, b, n_boot=2000, seed=0):
    """Mean of b - a over paired items (the same questions), with a 95% bootstrap interval over items."""
    rng = random.Random(seed)
    d = [y - x for x, y in zip(a, b)]
    means = sorted(sum(rng.choices(d, k=len(d))) / len(d) for _ in range(n_boot))
    return {"diff": mean(d), "lo": means[int(0.025 * n_boot)], "hi": means[int(0.975 * n_boot) - 1]}


def think_closed(text):
    """Did the episode close its first <think> block before its first <search> or <answer> tag?"""
    cut = min([i for i in (text.find("<search>"), text.find("<answer>")) if i >= 0], default=len(text))
    return "</think>" in text[:cut]


def failure_modes(agent_rows):
    """Tallies for the failure mode summary, per dataset: how episodes end, how many searches they use, how often
    a query is empty or repeats an earlier query of the same episode (repeat_query: the redundant search rate),
    how many distinct queries an episode uses, whether it reasons before its first tag, and F1 by search count."""
    out = {}
    for ds, v in by_dataset(agent_rows).items():
        queries = empty = repeat = 0
        distinct = []
        for r in v:
            seen = set()
            for s in r["searches"]:
                q = norm_query(s["query"])
                queries += 1
                empty += not q
                repeat += bool(q) and q in seen
                seen.add(q)
            distinct.append(len(seen))
        buckets = {"0": [r for r in v if r["n_search"] == 0], "1": [r for r in v if r["n_search"] == 1],
                   "2+": [r for r in v if r["n_search"] >= 2]}
        out[ds] = {"n": len(v), "stop": {k: c / len(v) for k, c in sorted(collections.Counter(
                       r["stop_reason"] for r in v).items())},
                   "n_search_share": {k: len(b) / len(v) for k, b in buckets.items()},
                   "f1_by_n_search": {k: mean(r["f1"] for r in b) for k, b in buckets.items() if b},
                   "empty_query": empty / max(queries, 1), "repeat_query": repeat / max(queries, 1),
                   "distinct_queries": mean(distinct),
                   "think_closed": mean(think_closed(r["text"]) for r in v if "text" in r)}
    return out

In [ ]:
%%writefile /content/code/export_site.py
"""Exports the recorded evaluation trajectories for the static demo page (docs/ on GitHub Pages).

Reads only: data/<set>_test.jsonl and outputs/<folder>/<task>.jsonl (with outputs/<folder>_ood/ for the three out
of domain sets), exactly the files the evaluation notebook read. Nothing is rerun. Writes:
  <out>/meta.json        policies, per policy metrics on every test set, the check against the report, showcase
  <out>/q_<set>.json     one file per test set: questions, a shared passage table, and every policy's trajectory

usage: python export_site.py --project P --out DIR [--limit N]
--limit keeps the first N questions per set in the q_ files (dry runs); metrics and checks always use every row.
"""
import argparse
import json
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from sot.analysis import by_dataset, failure_modes, load_jsonl, mean, qor, summarize
from sot.scoring import knows, norm_query

SETS = ["hotpotqa", "2wiki", "musique", "nq", "popqa"]
SET_LABELS = {"hotpotqa": "HotpotQA", "2wiki": "2WikiMultihopQA", "musique": "MuSiQue", "nq": "NQ", "popqa": "PopQA"}
IN_DOMAIN = {"hotpotqa", "nq"}

# key, outputs folder, task file, label, description. The four trained rewards all branch from the start policy.
POLICIES = [
    ("base_agent", "base_0.5B", "agent", "Untrained agent",
     "Qwen2.5-0.5B-Instruct prompted to use the search tags, before any RL"),
    ("start_agent", "stageA2", "agent", "Start policy",
     "Plain GRPO with forced exploration for 200 steps (stage A2). Every run below branches from it"),
    ("start_rag", "stageA2", "rag", "One search (RAG)",
     "The start policy given exactly one search, with the question itself as the query"),
    ("plain", "B_plain_s0", "agent", "Plain GRPO",
     "Reward: answer F1. 100 more steps from the start policy, seed 0"),
    ("fixed", "B_fixed05_s0", "agent", "Fixed penalty",
     "Reward: F1 minus 0.05 per search, on every rollout"),
    ("saas", "B_saas_s0", "agent", "SAAS",
     "Penalises searches its hard knowledge boundary labels call unnecessary, only when the answer is correct"),
    ("soft", "B_soft_s0", "agent", "Soft-SAAS (ours)",
     "SAAS with the boundary weighted by p(q), the start policy's closed book accuracy, seed 0"),
]
CLOSED = ("stageA2", "no_search")  # the start policy without search: shown as "closed book" on each question
PQ_FOLDER = "stageA2_pq"           # 8 closed book samples per test question: p(q), as in the report
KNOWN = 0.5                        # p(q) >= 0.5 counts as known, as for QOR in the report

# Report appendix (Table: every run), EM / F1 in % and SC, per set in SETS order. The export must reproduce these.
REPORT = {
    "base_agent": [(6.6, 13.9, 0.79), (7.4, 18.0, 0.96), (1.2, 4.8, 0.85), (10.6, 16.6, 0.75), (16.6, 21.2, 0.96)],
    "start_agent": [(15.1, 22.6, 2.54), (18.2, 23.1, 2.64), (2.2, 7.5, 2.61), (18.2, 25.0, 2.58), (24.0, 28.0, 2.57)],
    "start_rag": [(20.0, 28.7, 1.00), (21.8, 27.7, 1.00), (3.4, 9.1, 1.00), (17.2, 24.1, 1.00), (27.8, 30.9, 1.00)],
    "plain": [(19.3, 26.5, 3.00), (23.8, 27.8, 3.00), (3.6, 7.8, 3.00), (19.4, 28.1, 2.99), (25.4, 28.0, 3.00)],
    "fixed": [(10.6, 18.5, 0.00), (18.7, 23.8, 0.00), (1.8, 7.4, 0.00), (5.4, 12.0, 0.00), (11.0, 14.2, 0.00)],
    "saas": [(18.9, 26.9, 1.00), (24.5, 29.2, 1.00), (2.4, 7.4, 1.00), (18.8, 26.9, 1.00), (27.6, 31.3, 1.00)],
    "soft": [(19.6, 27.5, 1.00), (21.6, 27.2, 1.00), (2.2, 7.7, 1.00), (17.2, 25.1, 1.00), (24.4, 27.1, 1.00)],
    "closed": [(10.0, 17.7, 0.00), (16.8, 22.0, 0.00), (0.8, 6.5, 0.00), (7.6, 13.9, 0.00), (8.2, 14.3, 0.00)],
}
REPORT_KNOWN = {"hotpotqa": 102, "2wiki": 180, "nq": 31, "popqa": 57}  # report Table 4 (MuSiQue not reported)

INFO_RE = re.compile(r"<info>.*?</info>", re.S)


def rows_of(out, folder, task):
    rows = []
    for f in (folder, folder + "_ood"):
        path = f"{out}/{f}/{task}.jsonl"
        if os.path.exists(path):
            rows += load_jsonl(path)
    return rows


def model_text(text, cap):
    """What the model wrote: the environment's <info> blocks are cut (their passages are shown separately)."""
    t = INFO_RE.sub("<info>…</info>", text or "")
    return t if len(t) <= cap else t[:cap] + " …[cut]"


def repeats(queries):
    """How many queries repeat an earlier one of the same episode (case and spacing ignored)."""
    seen, n = set(), 0
    for q in map(norm_query, queries):
        n += bool(q) and q in seen
        seen.add(q)
    return n


def check_against_report(metrics):
    out, ok = {}, True
    for key, exp in REPORT.items():
        for ds, (e_em, e_f1, e_sc) in zip(SETS, exp):
            m = metrics.get(key, {}).get(ds)
            got = None if m is None else (100 * m["em"], 100 * m["f1"], m["sc"])
            # unrounded values against the report's rounded ones: within rounding (0.06 allows double rounding,
            # e.g. 22.545 -> 0.2255 -> 22.6 in the report)
            match = got is not None and abs(got[0] - e_em) <= 0.06 and abs(got[1] - e_f1) <= 0.06 \
                and abs(got[2] - e_sc) <= 0.006
            got = got and (round(got[0], 2), round(got[1], 2), round(got[2], 3))
            ok &= match
            if not match:
                out[f"{key}/{ds}"] = {"got": got, "report": (e_em, e_f1, e_sc)}
    return ok, out


def pick_showcase(qs, runs, per_kind=3):
    """Questions that show the report's findings, chosen by fixed rules from the recorded rows (test file order)."""
    picks, used = [], set()

    def take(kind, why, cond, sets=None, k=per_kind):
        n = 0
        for ds in (sets or SETS):
            for i, q in enumerate(qs[ds]):
                if n >= k:
                    return
                r = {key: runs[key][ds][i] for key in runs}
                if q["id"] in used or any(v is None for v in r.values()):
                    continue
                if cond(q, r):
                    picks.append({"id": q["id"], "set": ds, "kind": kind, "why": why})
                    used.add(q["id"])
                    n += 1
                    if sets is None:  # spread over the sets
                        break

    trained = ("plain", "fixed", "saas", "soft")
    take("repeats", "Plain GRPO repeats a query; SAAS searches once and gets it right",
         lambda q, r: repeats([x[0] for x in r["plain"]["s"]]) >= 1 and r["saas"]["n"] == 1 and r["saas"]["f1"] == 1.0,
         sets=["hotpotqa", "nq", "2wiki"], k=1)
    take("known", "The start policy knows this closed book, yet every trained policy still searches",
         lambda q, r: (q["p"] or 0) >= 0.75 and q["cb_ok"] and all(r[k]["n"] >= 1 for k in ("plain", "saas", "soft"))
         and min(r[k]["f1"] for k in ("plain", "saas", "soft")) < 0.5)
    take("fixed", "The fixed penalty never searches and misses; SAAS finds it with one search",
         lambda q, r: r["fixed"]["n"] == 0 and r["fixed"]["f1"] == 0 and r["saas"]["n"] == 1 and r["saas"]["f1"] == 1.0,
         sets=["hotpotqa", "popqa"], k=1)
    take("comparison", "A comparison question: one search for the whole question, often naming the wrong one",
         lambda q, r: bool(re.search(r"\bor\b", q["q"])) and bool(re.search(
             r"\b(first|earlier|older|younger|more recently|longer|larger)\b", q["q"])) and r["plain"]["n"] == 3,
         sets=["hotpotqa"], k=2)
    take("rag", "One search with the question beats every trained agent",
         lambda q, r: r["start_rag"]["f1"] == 1.0 and all(r[k]["f1"] == 0 for k in trained),
         sets=["hotpotqa", "popqa"], k=1)
    return picks


def build(project, out_dir, limit=None, passage_cap=500, text_cap=1500):
    data, out = f"{project}/data", f"{project}/outputs"
    recs = {}
    order = {}
    for ds in SETS:
        path = f"{data}/{ds}_test.jsonl"
        order[ds] = []
        for r in (load_jsonl(path) if os.path.exists(path) else []):
            recs[r["id"]] = r
            order[ds].append(r["id"])

    all_rows = {key: rows_of(out, folder, task) for key, folder, task, _, _ in POLICIES}
    all_rows["closed"] = rows_of(out, *CLOSED)
    by_id = {key: {r["id"]: r for r in rows} for key, rows in all_rows.items()}

    pq = {}
    for r in rows_of(out, PQ_FOLDER, "closed_test"):
        if r["id"] in recs:
            pq.setdefault(r["id"], []).append(knows(r["answer"], recs[r["id"]]["golds"]))
    pmap = {k: mean(v) for k, v in pq.items()}

    # Per policy metrics on every row (as in the report), and the check that they reproduce the report.
    metrics = {}
    for key, rows in all_rows.items():
        if not rows:
            continue
        fm = failure_modes(rows) if key != "closed" else {}
        qo = qor(rows, pmap, KNOWN) if pmap else {}
        metrics[key] = {ds: {"n": m["n"], "em": m["em"], "f1": m["f1"], "sc": m["searches"],
                             "format": m["format_ok"],
                             "repeat": fm.get(ds, {}).get("repeat_query"),
                             "distinct": fm.get(ds, {}).get("distinct_queries"),
                             "search_known": qo.get(ds, {}).get("qor"),
                             "search_rest": qo.get(ds, {}).get("search_rate_rest")}
                        for ds, m in summarize(rows).items()}
    report_ok, mismatches = check_against_report(metrics)
    known = {ds: sum(pmap.get(i, 0) >= KNOWN for i in order[ds] if i in pmap) for ds in SETS}
    known_ok = all(known.get(ds) == n for ds, n in REPORT_KNOWN.items())

    # Questions in test file order that every shown policy answered (the reference is the start policy).
    keys = [p[0] for p in POLICIES]
    qs, runs, passages, missing = {}, {k: {} for k in keys}, {}, {}
    os.makedirs(out_dir, exist_ok=True)
    for ds in SETS:
        ids = [i for i in order[ds] if i in by_id["start_agent"]]
        missing[ds] = {k: sum(i not in by_id[k] for i in ids) for k in keys + ["closed"]}
        missing[ds]["no_p"] = sum(i not in pmap for i in ids)
        table, index = [], {}

        def pid(title, text):
            k = (title, text[:passage_cap])
            if k not in index:
                index[k] = len(table)
                table.append([k[0], k[1] + ("…" if len(text) > passage_cap else "")])
            return index[k]

        qs[ds] = []
        for i in ids:
            rec, cb = recs[i], by_id["closed"].get(i)
            qs[ds].append({"id": i, "q": rec["question"], "golds": rec["golds"],
                           "p": round(pmap[i], 3) if i in pmap else None,
                           "cb": cb["answer"] if cb else None, "cb_ok": bool(cb) and knows(cb["answer"], rec["golds"]),
                           "cb_f1": round(cb["f1"], 3) if cb else None})
        for k in keys:
            runs[k][ds] = []
            for i in ids:
                r = by_id[k].get(i)
                if r is None:
                    runs[k][ds].append(None)
                    continue
                runs[k][ds].append({
                    "a": r["answer"], "f1": round(r["f1"], 3), "em": r["em"], "n": r["n_search"],
                    "stop": r["stop_reason"], "ok": r["format_ok"],
                    "s": [[s["query"], [pid(t, x) for t, x in zip(s.get("titles", []), s.get("texts", []))]]
                          for s in r.get("searches", [])],
                    "t": model_text(r.get("text", ""), text_cap)})
        passages[ds] = table

    showcase = pick_showcase(qs, runs)
    sizes = {}
    for ds in SETS:
        n = len(qs[ds]) if limit is None else min(limit, len(qs[ds]))
        payload = {"set": ds, "questions": qs[ds][:n], "runs": {k: runs[k][ds][:n] for k in keys}}
        if limit is not None:  # keep only the passages the cut questions use
            used = sorted({p for k in keys for r in payload["runs"][k] if r for s in r["s"] for p in s[1]})
            remap = {p: j for j, p in enumerate(used)}
            for k in keys:
                for r in payload["runs"][k]:
                    if r:
                        r["s"] = [[s[0], [remap[p] for p in s[1]]] for s in r["s"]]
            payload["passages"] = [passages[ds][p] for p in used]
        else:
            payload["passages"] = passages[ds]
        path = f"{out_dir}/q_{ds}.json"
        with open(path, "w") as f:
            json.dump(payload, f, ensure_ascii=False, separators=(",", ":"))
        sizes[ds] = round(os.path.getsize(path) / 1e6, 2)

    if limit is not None:  # showcase must point at questions that are in the cut files
        kept = {q["id"] for ds in SETS for q in qs[ds][:limit]}
        showcase = [s for s in showcase if s["id"] in kept]
    meta = {
        "title": "Search or Think? Soft Knowledge Boundaries for Cost-Aware Retrieval in Small Language Models",
        "model": "Qwen2.5-0.5B-Instruct",
        "sets": [{"key": ds, "label": SET_LABELS[ds], "in_domain": ds in IN_DOMAIN,
                  "n": len(qs[ds]) if limit is None else min(limit, len(qs[ds])), "known": known.get(ds)} for ds in SETS],
        "policies": [{"key": k, "label": lab, "desc": desc, "folder": folder, "task": task}
                     for k, folder, task, lab, desc in POLICIES],
        "metrics": {k: {ds: {m: (round(v, 4) if isinstance(v, float) else v) for m, v in d.items()}
                        for ds, d in v.items()} for k, v in metrics.items()},
        "known_threshold": KNOWN,
        "showcase": showcase,
        "check": {"report_match": report_ok, "known_counts_match": known_ok},
    }
    with open(f"{out_dir}/meta.json", "w") as f:
        json.dump(meta, f, ensure_ascii=False, separators=(",", ":"))
    sizes["meta"] = round(os.path.getsize(f"{out_dir}/meta.json") / 1e6, 3)

    return {"questions": {ds: len(qs[ds]) for ds in SETS}, "missing": missing, "known": known,
            "report_match": report_ok, "mismatches": mismatches, "known_counts_match": known_ok,
            "passages": {ds: len(passages[ds]) for ds in SETS}, "mb": sizes,
            "showcase": [(s["id"], s["kind"]) for s in showcase]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--passage_cap", type=int, default=500, help="characters kept per passage")
    ap.add_argument("--text_cap", type=int, default=1500, help="characters kept of what the model wrote")
    a = ap.parse_args()
    print(json.dumps(build(a.project, a.out, a.limit, a.passage_cap, a.text_cap)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/code/test_export_site.py
"""Tests for export_site.py on a tiny fake project with the same folder layout as the real Drive folder."""
import json
import os

import pytest

import export_site as ex

SETS = ex.SETS


def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)


def agent_row(rid, ds, answer, f1, queries, text=None, stop="answer"):
    searches = [{"query": q, "titles": [f"T {q}", "Shared"], "texts": [f"text for {q} " * 3, "same passage"]}
                for q in queries]
    return {"id": rid, "dataset": ds, "split": "test", "sample": 0, "answer": answer, "format_ok": True,
            "em": float(f1 == 1.0), "f1": f1, "n_search": len(queries), "stop_reason": stop,
            "n_distinct": len(set(queries)), "searches": searches,
            "text": text or "<think> x </think>" + "".join(
                f"<search> {q} </search>\n\n<info>\nDoc 1 (Title: T) long passage\n</info>\n\n" for q in queries)
            + f"<answer> {answer} </answer>"}


@pytest.fixture()
def project(tmp_path):
    p = tmp_path / "proj"
    recs = {ds: [{"id": f"{ds}_{i}", "dataset": ds, "split": "test", "question": f"Q{i} of {ds}: who or what?",
                  "golds": [f"gold{i}"]} for i in range(4)] for ds in SETS}
    recs["hotpotqa"][3]["question"] = "Who was born first, Ann Lee or Bo Ray?"
    for ds in SETS:
        write_jsonl(f"{p}/data/{ds}_test.jsonl", recs[ds] + [{"id": f"{ds}_extra", "dataset": ds, "split": "test",
                                                               "question": "not evaluated", "golds": ["x"]}])

    def run(folder, task, fn):  # in domain sets in outputs/<folder>/, the other three in outputs/<folder>_ood/
        by_sub = {}
        for ds in SETS:
            by_sub.setdefault(folder if ds in ex.IN_DOMAIN else folder + "_ood", []).extend(fn(ds, i) for i in range(4))
        for sub, rows in by_sub.items():
            write_jsonl(f"{p}/outputs/{sub}/{task}.jsonl", rows)

    run("stageA2", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, f"gold{i}", 1.0, ["q a", "q a"]))
    run("stageA2", "rag", lambda ds, i: agent_row(f"{ds}_{i}", ds, f"gold{i}" if i == 1 else "no", float(i == 1),
                                                  [recs[ds][i]["question"]]))
    run("stageA2", "no_search", lambda ds, i: agent_row(f"{ds}_{i}", ds, f"gold{i}" if i == 0 else "no",
                                                        float(i == 0), []))
    run("base_0.5B", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, "no", 0.0, ["b"]))
    run("B_plain_s0", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, "no" if i else "gold0", float(i == 0),
                                                       ["Q", "q", "Q"] if i != 2 else ["Q", "r", "s"]))
    run("B_fixed05_s0", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, "no", 0.0, []))
    run("B_saas_s0", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, f"gold{i}" if i != 1 else "no",
                                                      float(i != 1), ["one"]))
    run("B_soft_s0", "agent", lambda ds, i: agent_row(f"{ds}_{i}", ds, "no", 0.0, ["one"]))
    # p(q): 8 samples per question; question 0 known (6 of 8), the rest unknown
    write_jsonl(f"{p}/outputs/stageA2_pq/closed_test.jsonl",
                [{"id": f"{ds}_{i}", "dataset": ds, "answer": f"gold{i}" if (i == 0 and k < 6) else "no",
                  "em": 0.0, "f1": 0.0, "n_search": 0} for ds in SETS for i in range(4) for k in range(8)])
    return str(p)


def test_model_text_cuts_info_and_caps():
    t = "<think> a </think><search> q </search>\n\n<info>\nDoc 1 long\n</info>\n\n<answer> x </answer>"
    out = ex.model_text(t, 1000)
    assert "Doc 1" not in out and "<info>…</info>" in out and out.endswith("<answer> x </answer>")
    assert ex.model_text("abcdef", 3) == "abc …[cut]"


def test_repeats_counts_normalised_duplicates():
    assert ex.repeats(["A b", "a  B", "c", "A B"]) == 2
    assert ex.repeats(["", ""]) == 0  # empty queries are not repeats


def test_check_against_report_passes_on_exact_values_and_flags_others():
    metrics = {k: {ds: {"em": e / 100, "f1": f / 100, "sc": s} for ds, (e, f, s) in zip(SETS, v)}
               for k, v in ex.REPORT.items()}
    ok, bad = ex.check_against_report(metrics)
    assert ok and bad == {}
    metrics["saas"]["hotpotqa"]["f1"] += 0.002  # 0.2 points off
    ok, bad = ex.check_against_report(metrics)
    assert not ok and list(bad) == ["saas/hotpotqa"]
    del metrics["plain"]["nq"]
    ok, bad = ex.check_against_report(metrics)
    assert "plain/nq" in bad and bad["plain/nq"]["got"] is None


def test_build_writes_consistent_files(project, tmp_path):
    out = str(tmp_path / "site")
    rep = ex.build(project, out)
    assert rep["questions"] == {ds: 4 for ds in SETS}  # the not evaluated extra question is dropped
    assert all(v == 0 for m in rep["missing"].values() for v in m.values())
    assert rep["known"] == {ds: 1 for ds in SETS}
    assert rep["report_match"] is False and rep["known_counts_match"] is False  # fake numbers
    meta = json.load(open(f"{out}/meta.json"))
    assert [p["key"] for p in meta["policies"]] == [p[0] for p in ex.POLICIES]
    m = meta["metrics"]
    assert m["plain"]["hotpotqa"]["sc"] == 3 and m["fixed"]["nq"]["sc"] == 0
    assert m["start_agent"]["hotpotqa"]["repeat"] == 0.5  # every second query repeats the first
    assert m["saas"]["hotpotqa"]["f1"] == 0.75 and m["closed"]["hotpotqa"]["f1"] == 0.25
    assert m["saas"]["hotpotqa"]["search_known"] == 1.0  # searches the known question
    for ds in SETS:
        d = json.load(open(f"{out}/q_{ds}.json"))
        assert [q["id"] for q in d["questions"]] == [f"{ds}_{i}" for i in range(4)]
        q0 = d["questions"][0]
        assert q0["p"] == 0.75 and q0["cb"] == "gold0" and q0["cb_ok"] is True
        assert d["questions"][1]["cb_ok"] is False
        n = len(d["passages"])
        for k, rows in d["runs"].items():
            assert len(rows) == 4
            for r in rows:
                assert all(0 <= p < n for s in r["s"] for p in s[1])
                assert "long passage" not in r["t"]
        # "Shared" passage appears once in the table although every search retrieves it
        assert sum(1 for t, _ in d["passages"] if t == "Shared") == 1
        plain0 = d["runs"]["plain"][0]
        assert [s[0] for s in plain0["s"]] == ["Q", "q", "Q"] and plain0["n"] == 3


def test_showcase_rules_pick_expected_questions(project, tmp_path):
    rep = ex.build(project, str(tmp_path / "site"))
    kinds = {}
    for i, k in rep["showcase"]:
        kinds.setdefault(k, []).append(i)
    assert kinds == {"repeats": ["hotpotqa_0"],               # plain repeats "Q"; SAAS one search, correct
                     "known": ["2wiki_0", "musique_0", "nq_0"],  # p = 0.75 and closed book right; spread over sets
                     "fixed": ["hotpotqa_2"],                 # hotpotqa_0 is taken already
                     "comparison": ["hotpotqa_3"],
                     "rag": ["hotpotqa_1"]}
    ids = [i for i, _ in rep["showcase"]]
    assert len(ids) == len(set(ids))


def test_limit_keeps_valid_passage_indices_and_showcase(project, tmp_path):
    out = str(tmp_path / "dry")
    rep = ex.build(project, out, limit=2)
    meta = json.load(open(f"{out}/meta.json"))
    kept = set()
    for ds in SETS:
        d = json.load(open(f"{out}/q_{ds}.json"))
        assert len(d["questions"]) == 2
        kept |= {q["id"] for q in d["questions"]}
        n = len(d["passages"])
        used = {p for rows in d["runs"].values() for r in rows for s in r["s"] for p in s[1]}
        assert used == set(range(n))  # every kept passage is used, every index valid
    assert all(s["id"] in kept for s in meta["showcase"])
    assert meta["metrics"]["plain"]["hotpotqa"]["n"] == 4  # metrics still use every row
    assert rep["questions"]["hotpotqa"] == 4

## Unit tests

In [ ]:
!cd {CODE} && python -m pytest -q test_export_site.py

## Dry run

Three questions per test set into a scratch folder. Metrics and checks already use every row.

In [ ]:
!cd {CODE} && python export_site.py --project {PROJECT} --out /content/site_dry --limit 3
for f in sorted(os.listdir("/content/site_dry")):
    d = json.load(open(f"/content/site_dry/{f}"))
    print(f, {k: len(v) for k, v in d.items() if isinstance(v, (list, dict))})

## Export

In [ ]:
import importlib
import export_site
importlib.reload(export_site)

shutil.rmtree(SITE, ignore_errors=True)
REPORT = export_site.build(PROJECT, SITE)
shutil.make_archive("/content/site_data", "zip", SITE)
shutil.copy("/content/site_data.zip", f"{PROJECT}/site_data.zip")
print("zip:", round(os.path.getsize("/content/site_data.zip") / 1e6, 1), "MB, also saved to", f"{PROJECT}/site_data.zip")

## Results

In [ ]:
missing = {ds: {k: v for k, v in m.items() if v} for ds, m in REPORT["missing"].items()}
GATE = REPORT["report_match"] and REPORT["known_counts_match"] and not any(missing.values())
print(json.dumps({"gate": "PASS" if GATE else "FAIL", "questions": REPORT["questions"], "known": REPORT["known"],
                  "report_match": REPORT["report_match"], "mismatches": REPORT["mismatches"],
                  "known_counts_match": REPORT["known_counts_match"], "missing": missing,
                  "passages": REPORT["passages"], "mb": REPORT["mb"],
                  "zip_mb": round(os.path.getsize("/content/site_data.zip") / 1e6, 1),
                  "showcase": REPORT["showcase"]}))

## Download

Your browser downloads `site_data.zip` (if it asks, allow downloads from colab). The same zip is in the Drive project folder.

In [ ]:
from google.colab import files
files.download("/content/site_data.zip")

In [ ]:
import time
time.sleep(60)  # shortcut: gives the browser a minute to finish the download before the runtime is released
from google.colab import runtime
runtime.unassign()